# ThreadCraft — Size/Fit Recommender · Step 1: Data Cleaning

**Run on Kaggle with the CPU accelerator** — no GPU needed here or in step 2. This whole model trains on CPU in minutes, which is exactly why it's the *reliable* ML deliverable for this project.

**Source:** the Clothing Fit Dataset (Misra, Wan & McAuley, RecSys 2018), **RentTheRunway** split — 192,544 real rental transactions where a customer recorded their body measurements, the size they took, and whether it ran **small / fit / large**.

Downloaded **directly from the UCSD McAuley Lab**, which needs no Kaggle dataset attachment and no login:
```
https://mcauleylab.ucsd.edu/public_datasets/data/renttherunway/renttherunway_final_data.json.gz
```

We use RentTheRunway rather than the ModCloth split in the same release because RentTheRunway's body-measurement fields are 84–92% populated, where ModCloth's are as low as 3.5% (`waist`) and 14.3% (`bust`) — effectively unusable.

**Licence:** CC BY 4.0. Cite Misra, Wan & McAuley (RecSys 2018).

## How this maps onto ThreadCraft

The dataset answers: *given this person's body and the size they took, did it fit?*

ThreadCraft needs the inverse: *given this person's body, what size should we cut?*

So the model is trained as a **fit classifier** `P(small | fit | large  |  body, size)`, and at inference time the API sweeps candidate sizes and picks the one maximising `P(fit)`. That inversion is done in the API layer — see `02_train.ipynb` for the demonstration and `docs/` for the write-up.

### Features deliberately excluded (to avoid leakage)

| Excluded | Why |
|---|---|
| `rating`, `review_text`, `review_summary` | Written **after** wearing the garment. Strongly correlated with fit, and unavailable at prediction time. Including them would inflate the reported metrics with information the deployed system can never have. |
| `user_id` | Not available for a new ThreadCraft customer, and would leak per-user fit habits across the train/test split. |
| `item_id` | ThreadCraft makes bespoke garments; there is no catalogue item to look up. |

Everything kept is something a ThreadCraft customer actually supplies in the wizard.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
HF_USERNAME = "your-hf-username"  # <-- CHANGE THIS to your Hugging Face username

CLEANED_REPO_ID = f"{HF_USERNAME}/threadcraft-fit-cleaned"
SOURCE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/renttherunway/renttherunway_final_data.json.gz"

VAL_FRACTION = 0.10
TEST_FRACTION = 0.10
RANDOM_SEED = 42
PUSH_TO_HUB = True

# Plausibility bounds — values outside these become NaN rather than being dropped,
# so the row's other (valid) features are still usable.
AGE_MIN, AGE_MAX = 10, 100
HEIGHT_CM_MIN, HEIGHT_CM_MAX = 120, 210
WEIGHT_KG_MIN, WEIGHT_KG_MAX = 30, 200

In [ ]:
# Kaggle already ships pandas / numpy / scikit-learn / pyarrow / joblib.
# Deliberately NOT upgrading them: '-U pandas' pulls 3.x, which conflicts
# with Kaggle's preinstalled gradio and risks breakage for no benefit here.
!pip install -q -U datasets huggingface_hub

In [ ]:
import os

from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"Not on Kaggle or secret missing ({e}). Falling back to the HF_TOKEN env var.")
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
elif PUSH_TO_HUB:
    raise RuntimeError("No HF_TOKEN available but PUSH_TO_HUB is True.")

## 1. Download and load

The file is JSON **Lines** (one object per line), not a JSON array — `pd.read_json` needs `lines=True`.

In [ ]:
import gzip
import json
import urllib.request

LOCAL_GZ = "renttherunway.json.gz"

if not os.path.exists(LOCAL_GZ):
    print(f"Downloading {SOURCE_URL} ...")
    urllib.request.urlretrieve(SOURCE_URL, LOCAL_GZ)
print(f"Size on disk: {os.path.getsize(LOCAL_GZ):,} bytes")

records = []
with gzip.open(LOCAL_GZ, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"Loaded {len(records):,} records")

In [ ]:
import pandas as pd

df = pd.DataFrame(records)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head(3)

In [ ]:
print("Missing values per column:")
missing = df.isnull().sum().sort_values(ascending=False)
for col, n in missing.items():
    print(f"  {col:20s} {n:7,}  ({n / len(df) * 100:5.2f}%)")

print("\nTarget distribution ('fit'):")
print(df["fit"].value_counts())
print((df["fit"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

## 2. Parse the messy string fields

Three fields arrive as human-typed strings and need real parsing:

| Field | Raw form | Parsed to |
|---|---|---|
| `height` | `5' 8"` | `height_cm` (float) |
| `weight` | `137lbs` | `weight_kg` (float) |
| `bust size` | `34d` | `bust_band` (int) + `bust_cup` (ordinal int) |

Every parser returns `NaN` on anything unparseable rather than raising — the model handles missing values natively, so one malformed row shouldn't cost us the other 9 usable features in it.

In [ ]:
import re

import numpy as np


def parse_height_cm(value):
    """5' 8\" -> 172.72"""
    if not isinstance(value, str):
        return np.nan
    m = re.match(r"\s*(\d+)\s*'\s*(\d+)?", value)
    if not m:
        return np.nan
    feet = int(m.group(1))
    inches = int(m.group(2)) if m.group(2) else 0
    return round((feet * 12 + inches) * 2.54, 2)


def parse_weight_kg(value):
    """137lbs -> 62.14"""
    if not isinstance(value, str):
        return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)", value)
    if not m:
        return np.nan
    return round(float(m.group(1)) * 0.45359237, 2)


# Cup letters are ordinal (a < b < c ...), so they get an ordinal encoding rather
# than one-hot: the ordering carries real information about bust volume.
CUP_ORDER = ["aa", "a", "b", "c", "d", "dd", "ddd/e", "f", "g", "h", "i", "j"]
CUP_TO_NUM = {cup: i for i, cup in enumerate(CUP_ORDER)}


def parse_bust(value):
    """34d -> (34, 4)"""
    if not isinstance(value, str):
        return (np.nan, np.nan)
    m = re.match(r"\s*(\d+)\s*([a-zA-Z/]+)\s*$", value.strip().lower())
    if not m:
        return (np.nan, np.nan)
    band = int(m.group(1))
    cup = CUP_TO_NUM.get(m.group(2), np.nan)
    return (band, cup)


def parse_numeric(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


# Quick self-check on the parsers before running them over 192k rows.
assert parse_height_cm("5' 8\"") == 172.72
assert parse_height_cm("6' 0\"") == 182.88
assert np.isnan(parse_height_cm(None))
assert parse_weight_kg("137lbs") == 62.14
assert np.isnan(parse_weight_kg(""))
assert parse_bust("34d") == (34, 4)
assert parse_bust("32aa") == (32, 0)
print("Parser self-checks passed.")

In [ ]:
clean = pd.DataFrame()

clean["fit"] = df["fit"]
clean["height_cm"] = df["height"].apply(parse_height_cm)
clean["weight_kg"] = df["weight"].apply(parse_weight_kg)

bust_parsed = df["bust size"].apply(parse_bust)
clean["bust_band"] = [b[0] for b in bust_parsed]
clean["bust_cup"] = [b[1] for b in bust_parsed]

clean["age"] = df["age"].apply(parse_numeric)
clean["size"] = df["size"].apply(parse_numeric)
clean["body_type"] = df["body type"]
clean["category"] = df["category"]
clean["rented_for"] = df["rented for"]

print(clean.dtypes)
clean.head()

## 3. Handle outliers and inconsistent categories

Real data problems found while profiling this dataset:
- **65 rows report an age over 100** (max 117) and some report age 0 — clearly form-entry errors
- **`rented for` contains a single `'party: cocktail'` row** — a typo variant of `party`

Out-of-range numerics become `NaN` (keeping the row's other features) rather than dropping the row.

In [ ]:
def clip_to_nan(series, lo, hi, name):
    bad = ((series < lo) | (series > hi)) & series.notna()
    if bad.sum():
        print(f"  {name}: {bad.sum():,} implausible values -> NaN "
              f"(range seen: {series.min():.0f}–{series.max():.0f}, kept: {lo}–{hi})")
    return series.mask(bad)


print("Outlier handling:")
clean["age"] = clip_to_nan(clean["age"], AGE_MIN, AGE_MAX, "age")
clean["height_cm"] = clip_to_nan(clean["height_cm"], HEIGHT_CM_MIN, HEIGHT_CM_MAX, "height_cm")
clean["weight_kg"] = clip_to_nan(clean["weight_kg"], WEIGHT_KG_MIN, WEIGHT_KG_MAX, "weight_kg")

In [ ]:
print("'rented_for' before normalisation:")
print(clean["rented_for"].value_counts(dropna=False))

# Collapse the single 'party: cocktail' typo into 'party'.
clean["rented_for"] = clean["rented_for"].replace({"party: cocktail": "party"})

print("\nAfter:")
print(clean["rented_for"].value_counts(dropna=False))

In [ ]:
# BMI: a single number capturing the height/weight relationship, which is more
# directly predictive of fit than either dimension alone.
clean["bmi"] = (clean["weight_kg"] / (clean["height_cm"] / 100) ** 2).round(2)
clean["bmi"] = clip_to_nan(clean["bmi"], 12, 60, "bmi")

print(clean[["height_cm", "weight_kg", "bmi", "bust_band", "bust_cup", "age", "size"]].describe().round(2))

In [ ]:
# The target itself must never be missing.
before = len(clean)
clean = clean[clean["fit"].notna() & clean["fit"].isin(["small", "fit", "large"])]
print(f"Rows with a valid target: {before:,} -> {len(clean):,}")

# A row with no body information at all cannot contribute anything.
body_cols = ["height_cm", "weight_kg", "bust_band", "bust_cup", "bmi"]
before = len(clean)
clean = clean[clean[body_cols].notna().any(axis=1)]
print(f"Rows with >=1 body measurement: {before:,} -> {len(clean):,}")

clean = clean.reset_index(drop=True)
print(f"\nFinal cleaned shape: {clean.shape}")

In [ ]:
print("Remaining missing values (the model handles these natively — no imputation):")
miss = clean.isnull().sum().sort_values(ascending=False)
for col, n in miss[miss > 0].items():
    print(f"  {col:14s} {n:7,}  ({n / len(clean) * 100:5.2f}%)")
if miss.sum() == 0:
    print("  none")

## 4. Exploratory figures for the dissertation

In [ ]:
import matplotlib.pyplot as plt

PALETTE = {"small": "#C4A882", "fit": "#8B6B4A", "large": "#2C1F14"}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

counts = clean["fit"].value_counts()
axes[0].bar(counts.index, counts.values, color=[PALETTE[c] for c in counts.index])
axes[0].set_title(f"Target distribution (n={len(clean):,})")
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v, f"{v:,}\n{v / len(clean) * 100:.1f}%", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(0, counts.max() * 1.18)

for label in ["small", "fit", "large"]:
    subset = clean[clean["fit"] == label]["bmi"].dropna()
    axes[1].hist(subset, bins=45, alpha=0.55, label=label, color=PALETTE[label], density=True)
axes[1].set_title("BMI distribution by fit outcome")
axes[1].set_xlabel("BMI")
axes[1].legend()

for label in ["small", "fit", "large"]:
    subset = clean[clean["fit"] == label]["size"].dropna()
    axes[2].hist(subset, bins=30, alpha=0.55, label=label, color=PALETTE[label], density=True)
axes[2].set_title("Ordered size by fit outcome")
axes[2].set_xlabel("size")
axes[2].legend()

plt.tight_layout()
plt.savefig("fit_eda.png", dpi=130)
plt.show()

## 5. Stratified split

Split **here**, in the cleaning notebook, and ship the split with the dataset — so the training notebook cannot accidentally reshuffle and leak, and any re-run reproduces exactly the same partition.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    clean,
    test_size=VAL_FRACTION + TEST_FRACTION,
    stratify=clean["fit"],
    random_state=RANDOM_SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    stratify=temp_df["fit"],
    random_state=RANDOM_SEED,
)

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    dist = (part["fit"].value_counts(normalize=True) * 100).round(2).to_dict()
    print(f"{name:11s} {len(part):7,}  {dist}")

## 6. Push to the Hub

In [ ]:
from datasets import Dataset, DatasetDict

dataset_dict = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
    }
)
dataset_dict

In [ ]:
if PUSH_TO_HUB:
    try:
        dataset_dict.push_to_hub(CLEANED_REPO_ID, private=False)
        print(f"Pushed: https://huggingface.co/datasets/{CLEANED_REPO_ID}")
    except Exception as e:
        print(f"PUSH FAILED: {e}")
        for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
            part.to_parquet(f"/kaggle/working/fit_{name}.parquet")
        print("Saved parquet files to /kaggle/working instead — re-push from there.")
else:
    for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
        part.to_parquet(f"fit_{name}.parquet")
    print("PUSH_TO_HUB is False — parquet files written locally.")

## Summary for the dissertation

In [ ]:
print("=" * 62)
print("DATA PREPARATION SUMMARY — size/fit recommender")
print("=" * 62)
print("Source          : Clothing Fit Dataset (Misra, Wan & McAuley, RecSys 2018)")
print("Split used      : RentTheRunway")
print("Licence         : CC BY 4.0")
print(f"Raw records     : {len(df):,}")
print(f"After cleaning  : {len(clean):,}")
print(f"Train/Val/Test  : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"Target          : fit in {{small, fit, large}}")
dist = clean["fit"].value_counts(normalize=True).mul(100).round(2).to_dict()
print(f"Class balance   : {dist}  <- majority class ~74%, so report macro-F1")
print("Features        : height_cm, weight_kg, bmi, bust_band, bust_cup, age,")
print("                  size, body_type, category, rented_for")
print("Excluded (leak) : rating, review_text, review_summary, user_id, item_id")
print("=" * 62)
print("\nNext: run 02_train.ipynb (CPU is fine — no GPU needed).")